In [1]:
"""
CarbonTatva AI — Model: Ridge Regression + PCA + Residual Analysis
===================================================================
Market : GDAM only
Data   : Last-year (Mar 2025 – Feb 2026)
Split  : Train ≤ Sep-2025  |  Val ≤ Nov-2025  |  Test > Nov-2025

Changes vs previous version:
  1. MARKETS = ['GDAM'] only
  2. PCA applied after StandardScaler — variance threshold 0.995
  3. Residual analysis block added (same as XGBoost version):
     - hourly bias, price-bucket bias, weekday bias, autocorrelation
"""

import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings('ignore')
OUT = 'outputs_ridge_pca'
os.makedirs(OUT, exist_ok=True)

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
DATA_FILE       = 'final_merged_data.xlsx'
MARKETS         = ['GDAM']
SLOTS_DAY       = 96
LAST_YEAR_START = '2025-03-01'
TRAIN_END       = '2025-09-30'
VAL_END         = '2025-11-30'
PCA_VAR         = 0.995
SEED            = 42
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────
def build_features(df):
    df = df.copy()
    d = pd.to_datetime(df['Delivery Date'], dayfirst=True)
    t = df['Time Period'].str.split('-').str[0]
    df['datetime'] = pd.to_datetime(d.astype(str) + ' ' + t)
    df = df.sort_values('datetime').reset_index(drop=True)

    df.drop(columns=['apparent_temperature (°C)',
                     'sunshine_duration (s)',
                     'precipitation (mm)'], errors='ignore', inplace=True)

    df['hour']  = df['datetime'].dt.hour
    df['dow']   = df['datetime'].dt.dayofweek
    df['month'] = df['datetime'].dt.month

    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)
    df['is_weekend'] = (df['dow'] >= 5).astype(int)

    try:
        import holidays
        india_h = holidays.India(years=range(2022, 2027))
        df['holiday_flag'] = df['datetime'].dt.date.apply(
            lambda d: 1 if d in india_h else 0)
    except ImportError:
        df['holiday_flag'] = 0

    p = df['Price (Rs./MWh)']
    df['price_lag96']  = p.shift(96)
    df['price_lag672'] = p.shift(672)
    df['ma3d']  = p.shift(1).rolling(288).mean()
    df['ma7d']  = p.shift(1).rolling(672).mean()
    df['ma14d'] = p.shift(1).rolling(1344).mean()
    df['price_diff1d'] = p.shift(96)  - p.shift(192)
    df['price_diff1w'] = p.shift(672) - p.shift(1344)
    df['price_level_ratio'] = df['price_lag96'] / (df['ma7d'] + 1e-6)
    df['rel_lag672']        = df['price_lag672'] / (df['ma7d'] + 1e-6)

    df['temp']    = df['temperature_2m (°C)']
    df['temp_sq'] = df['temp'] ** 2
    df['temp_cb'] = df['temp'] ** 3
    df['radiation']            = df['shortwave_radiation (W/m²)']
    df['radiation_x_hour_sin'] = df['radiation'] * df['hour_sin']
    df['radiation_x_hour_cos'] = df['radiation'] * df['hour_cos']
    df['radiation_sq']         = df['radiation'] ** 2
    df['wind_speed']  = df['wind_speed_100m (km/h)']
    df['wind_sq']     = df['wind_speed'] ** 2
    df['cloud_cover'] = df['cloud_cover (%)']
    df['humidity']    = df['relative_humidity_2m (%)']
    df['rain']        = df['rain (mm)']
    df['temp_x_hour_sin']   = df['temp'] * df['hour_sin']
    df['temp_x_is_weekend'] = df['temp'] * df['is_weekend']
    df['wind_x_radiation']  = df['wind_speed'] * df['radiation']

    return df


FEATURES = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'dow_sin', 'dow_cos', 'is_weekend', 'holiday_flag',
    'price_lag96', 'price_lag672', 'ma3d', 'ma7d', 'ma14d',
    'price_diff1d', 'price_diff1w',
    'price_level_ratio', 'rel_lag672',
    'temp', 'temp_sq', 'temp_cb',
    'radiation', 'radiation_sq', 'radiation_x_hour_sin', 'radiation_x_hour_cos',
    'wind_speed', 'wind_sq', 'cloud_cover', 'humidity', 'rain',
    'temp_x_hour_sin', 'temp_x_is_weekend', 'wind_x_radiation',
]

# ─────────────────────────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, label=''):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    print(f"    {label:20s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:.2f}%")
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def save_pred_plot(y_true, y_pred, title, path):
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27'); ax.tick_params(colors='#aaa')
    n = min(800, len(y_true))
    axes[0].plot(y_true[:n], color='#aaa', lw=0.8, label='Actual')
    axes[0].plot(y_pred[:n], color='#8e44ad', lw=0.8, label='Predicted')
    axes[0].set_title(f'{title} — Test Set', color='white', fontsize=12)
    axes[0].legend(fontsize=9); axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')
    n2 = SLOTS_DAY * 3
    axes[1].plot(y_true[:n2], color='#aaa', lw=1.5, label='Actual')
    axes[1].plot(y_pred[:n2], color='#8e44ad', lw=1.5, ls='--', label='Predicted')
    axes[1].set_title(f'{title} — First 3 Days', color='white', fontsize=12)
    axes[1].legend(fontsize=9); axes[1].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[2].scatter(y_true[:3000], y_pred[:3000], s=2, alpha=0.3, color='#8e44ad')
    lim = max(y_true[:3000].max(), y_pred[:3000].max())
    axes[2].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect')
    axes[2].set_title(f'{title} — Actual vs Predicted', color='white', fontsize=12)
    axes[2].set_xlabel('Actual', color='#aaa'); axes[2].set_ylabel('Predicted', color='#aaa')
    axes[2].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()


# ─────────────────────────────────────────────────────────────────
# RESIDUAL ANALYSIS
# ─────────────────────────────────────────────────────────────────
def residual_analysis(test_df, y_true, y_pred, mkt, out_dir):
    df = test_df.copy().reset_index(drop=True)
    df['residual'] = y_pred - y_true
    df['abs_err']  = np.abs(df['residual'])
    df['hour']     = df['datetime'].dt.hour
    df['dow']      = df['datetime'].dt.dayofweek
    df['date']     = df['datetime'].dt.date

    q33, q67    = np.percentile(y_true, 33), np.percentile(y_true, 67)
    spike_thr   = np.percentile(y_true, 95)
    def price_bucket(p):
        if p < q33:         return '① Low (<P33)'
        elif p < q67:       return '② Mid (P33-P67)'
        elif p < spike_thr: return '③ High (P67-P95)'
        else:               return '④ Spike (>P95)'
    df['bucket'] = y_true
    df['bucket'] = df['bucket'].apply(price_bucket)

    day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    bkt_order = ['① Low (<P33)', '② Mid (P33-P67)', '③ High (P67-P95)', '④ Spike (>P95)']

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor('#0f1117')
    fig.suptitle(f'Residual Analysis — Ridge+PCA GDAM\n'
                 f'(+ve = overestimate, –ve = underestimate)', color='white', fontsize=13)
    for ax in axes.flat:
        ax.set_facecolor('#1a1d27'); ax.tick_params(colors='#aaa')
        ax.axhline(0, color='white', lw=0.8, ls='--', alpha=0.5)

    # Plot 1 — hourly bias
    hrly = df.groupby('hour')['residual'].mean()
    colors_hr = ['#e74c3c' if v > 0 else '#2980b9' for v in hrly.values]
    axes[0,0].bar(hrly.index, hrly.values, color=colors_hr, alpha=0.85)
    axes[0,0].set_title('Bias by Hour of Day', color='white')
    axes[0,0].set_xlabel('Hour', color='#aaa')
    axes[0,0].set_ylabel('Mean Residual (Rs./MWh)', color='#aaa')
    axes[0,0].set_xticks(range(0, 24, 2))
    worst_h = hrly.abs().nlargest(3).index
    for h in worst_h:
        axes[0,0].annotate(f'{hrly[h]:+.0f}', xy=(h, hrly[h]),
                           xytext=(h, hrly[h] + (30 if hrly[h]>0 else -30)),
                           color='yellow', fontsize=8, ha='center')

    # Plot 2 — price bucket
    bkt_bias = df.groupby('bucket')['residual'].mean().reindex(bkt_order)
    bkt_mae  = df.groupby('bucket')['abs_err'].mean().reindex(bkt_order)
    x = np.arange(len(bkt_order)); w = 0.4
    colors_bkt = ['#e74c3c' if v > 0 else '#2980b9' for v in bkt_bias.values]
    axes[0,1].bar(x - w/2, bkt_bias.values, w, color=colors_bkt, alpha=0.85, label='Bias')
    axes[0,1].bar(x + w/2, bkt_mae.values,  w, color='#f39c12',  alpha=0.7,  label='MAE')
    axes[0,1].set_title('Bias & MAE by Price Regime', color='white')
    axes[0,1].set_xticks(x)
    axes[0,1].set_xticklabels([b.split('(')[0].strip() for b in bkt_order],
                               rotation=15, ha='right', fontsize=9)
    axes[0,1].set_ylabel('Rs./MWh', color='#aaa'); axes[0,1].legend(fontsize=9)
    for i, (b, m) in enumerate(zip(bkt_bias.values, bkt_mae.values)):
        axes[0,1].text(i-w/2, b + (15 if b>=0 else -25), f'{b:+.0f}', ha='center',
                       color='white', fontsize=8)
        axes[0,1].text(i+w/2, m+5, f'{m:.0f}', ha='center', color='white', fontsize=8)

    # Plot 3 — weekday
    dow_bias = df.groupby('dow')['residual'].mean()
    dow_mae  = df.groupby('dow')['abs_err'].mean()
    x3 = np.arange(7)
    colors_dow = ['#e74c3c' if v > 0 else '#2980b9' for v in dow_bias.values]
    axes[1,0].bar(x3 - w/2, dow_bias.values, w, color=colors_dow, alpha=0.85, label='Bias')
    axes[1,0].bar(x3 + w/2, dow_mae.values,  w, color='#f39c12',  alpha=0.7,  label='MAE')
    axes[1,0].set_title('Bias & MAE by Weekday', color='white')
    axes[1,0].set_xticks(x3); axes[1,0].set_xticklabels(day_names)
    axes[1,0].set_ylabel('Rs./MWh', color='#aaa'); axes[1,0].legend(fontsize=9)
    axes[1,0].axvspan(4.5, 6.5, color='white', alpha=0.05)
    axes[1,0].text(5.5, axes[1,0].get_ylim()[1]*0.9, 'Weekend',
                   color='#aaa', ha='center', fontsize=8)

    # Plot 4 — autocorrelation
    res = df['residual'].values
    max_lag = 48
    acf_vals = [np.corrcoef(res[:-lag], res[lag:])[0,1] for lag in range(1, max_lag+1)]
    ci = 1.96 / np.sqrt(len(res))
    colors_acf = ['#e74c3c' if abs(v) > ci else '#555' for v in acf_vals]
    axes[1,1].bar(range(1, max_lag+1), acf_vals, color=colors_acf, alpha=0.85)
    axes[1,1].axhline( ci, color='yellow', lw=1, ls='--', label=f'95% CI (±{ci:.3f})')
    axes[1,1].axhline(-ci, color='yellow', lw=1, ls='--')
    axes[1,1].set_title('Residual Autocorrelation (Lags 1–48 slots)', color='white')
    axes[1,1].set_xlabel('Lag (15-min slots)', color='#aaa')
    axes[1,1].set_ylabel('Correlation', color='#aaa')
    axes[1,1].legend(fontsize=9)
    sig_lags = [i+1 for i, v in enumerate(acf_vals) if abs(v) > ci]
    if sig_lags:
        axes[1,1].text(max_lag*0.6, 0.8*max(acf_vals),
                       f'Significant: {sig_lags[:5]}{"…" if len(sig_lags)>5 else ""}',
                       color='yellow', fontsize=8)

    plt.tight_layout()
    plt.savefig(f'{out_dir}/residual_analysis_{mkt}.png',
                dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()
    print(f"  Saved: {out_dir}/residual_analysis_{mkt}.png")

    day_mae = df.groupby('date')['abs_err'].mean().nlargest(10)
    print(f"\n  Top-10 worst predicted days (by MAE):")
    for dt, mae_val in day_mae.items():
        sub = df[df['date'] == dt]
        bias_val = sub['residual'].mean()
        direction = 'OVER' if bias_val > 0 else 'UNDER'
        print(f"    {dt}  MAE={mae_val:.0f}  avg_bias={bias_val:+.0f} ({direction})")

    print(f"\n  ── Residual Summary ──────────────────────────────────")
    print(f"  Overall bias (mean residual): {df['residual'].mean():+.1f} Rs./MWh")
    print(f"  Most overestimated hour : {hrly.idxmax():02d}:00  ({hrly.max():+.0f})")
    print(f"  Most underestimated hour: {hrly.idxmin():02d}:00  ({hrly.min():+.0f})")
    print(f"  Highest MAE price regime: {bkt_mae.idxmax()}  (MAE={bkt_mae.max():.0f})")
    sig_pct = 100 * len(sig_lags) / max_lag
    print(f"  Autocorrelated lags     : {len(sig_lags)}/{max_lag} ({sig_pct:.0f}%) — "
          + ("serial structure remains" if len(sig_lags) > 5 else "mostly white noise ✓"))


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("MODEL: Ridge Regression + PCA (Last Year Data)")
print(f"Market: GDAM only")
print(f"Period: {LAST_YEAR_START} onwards")
print(f"Split:  Train≤{TRAIN_END}  |  Val≤{VAL_END}  |  Test>Val")
print(f"PCA variance threshold: {PCA_VAR}")
print("=" * 60)

sheets = pd.read_excel(DATA_FILE, sheet_name=None)

for mkt in MARKETS:
    print(f"\n{'─'*50}\nMarket: {mkt}\n{'─'*50}")

    df_full = build_features(sheets[mkt])
    df_full = df_full.dropna(subset=FEATURES + ['Price (Rs./MWh)']).copy()
    df = df_full[df_full['datetime'] >= LAST_YEAR_START].copy()
    print(f"  Rows: {len(df):,}  ({df['datetime'].min().date()} → {df['datetime'].max().date()})")

    train = df[df['datetime'] <= TRAIN_END]
    val   = df[(df['datetime'] > TRAIN_END) & (df['datetime'] <= VAL_END)]
    test  = df[df['datetime'] > VAL_END]
    print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
    print(f"  Price mean — Train: {train['Price (Rs./MWh)'].mean():.0f}"
          f"  Val: {val['Price (Rs./MWh)'].mean():.0f}"
          f"  Test: {test['Price (Rs./MWh)'].mean():.0f}")

    X_tr = train[FEATURES].values; y_tr = train['Price (Rs./MWh)'].values
    X_va = val[FEATURES].values;   y_va = val['Price (Rs./MWh)'].values
    X_te = test[FEATURES].values;  y_te = test['Price (Rs./MWh)'].values

    # ── StandardScaler ────────────────────────────────────────────
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    X_te_s = scaler.transform(X_te)

    # ── PCA ───────────────────────────────────────────────────────
    pca = PCA(n_components=PCA_VAR, random_state=SEED)
    X_tr_p = pca.fit_transform(X_tr_s)
    X_va_p = pca.transform(X_va_s)
    X_te_p = pca.transform(X_te_s)
    n_comp = pca.n_components_
    var_exp = pca.explained_variance_ratio_.cumsum()[-1] * 100
    print(f"\n  PCA: {len(FEATURES)} features → {n_comp} components ({var_exp:.2f}% variance)")

    # PCA variance plot
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d27')
    ax.bar(range(1, n_comp+1), pca.explained_variance_ratio_ * 100,
           color='#8e44ad', alpha=0.85)
    ax.plot(range(1, n_comp+1), pca.explained_variance_ratio_.cumsum() * 100,
            color='#27ae60', lw=2, marker='o', ms=3, label='Cumulative')
    ax.axhline(99.5, color='yellow', lw=1, ls='--', label='99.5% threshold')
    ax.set_title(f'PCA Explained Variance — {mkt}', color='white')
    ax.set_xlabel('Component', color='#aaa'); ax.set_ylabel('Variance Explained (%)', color='#aaa')
    ax.legend(fontsize=9); ax.tick_params(colors='#aaa')
    plt.tight_layout()
    plt.savefig(f'{OUT}/pca_variance_{mkt}.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    # ── Alpha tuning on val ───────────────────────────────────────
    print("  Tuning alpha on PCA-transformed val set...")
    best_alpha, best_mae = None, float('inf')
    for alpha in [0.001, 0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0, 1000.0, 5000.0, 10000.0]:
        m = Ridge(alpha=alpha, max_iter=10000)
        m.fit(X_tr_p, y_tr)
        vm = mean_absolute_error(y_va, m.predict(X_va_p))
        if vm < best_mae:
            best_mae, best_alpha = vm, alpha
    print(f"  Best alpha: {best_alpha}  |  Val MAE: {best_mae:.1f}")

    # Refit on train+val
    X_tv = np.vstack([X_tr_p, X_va_p])
    y_tv = np.concatenate([y_tr, y_va])
    final_model = Ridge(alpha=best_alpha, max_iter=10000)
    final_model.fit(X_tv, y_tv)

    val_pred  = np.clip(final_model.predict(X_va_p), 0, 20000)
    test_pred = np.clip(final_model.predict(X_te_p), 0, 20000)

    print("  Results:")
    m_val  = compute_metrics(y_va, val_pred,  'Validation')
    m_test = compute_metrics(y_te, test_pred, 'Test')

    # PC coefficient plot
    idx = np.argsort(np.abs(final_model.coef_))
    bar_c = ['#e74c3c' if c > 0 else '#2980b9' for c in final_model.coef_[idx]]
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d27')
    pc_labels = [f'PC{i+1}' for i in range(n_comp)]
    ax.barh([pc_labels[i] for i in idx], final_model.coef_[idx], color=bar_c, alpha=0.85)
    ax.axvline(0, color='white', lw=0.8, ls='--')
    ax.set_title(f'Ridge Coefficients on PCs — {mkt}\nRed=+price  Blue=-price', color='white')
    ax.tick_params(colors='#aaa')
    plt.tight_layout()
    plt.savefig(f'{OUT}/ridge_coefs_{mkt}.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    save_pred_plot(y_te, test_pred, f'Ridge+PCA — {mkt}',
                   f'{OUT}/ridge_pred_{mkt}.png')

    # ── RESIDUAL ANALYSIS ─────────────────────────────────────────
    print(f"\n  Running residual analysis...")
    residual_analysis(test, y_te, test_pred, mkt, OUT)

    with open(f'{OUT}/ridge_pipeline_{mkt}.pkl', 'wb') as f:
        pickle.dump({'model': final_model, 'scaler': scaler, 'pca': pca,
                     'features': FEATURES, 'best_alpha': best_alpha,
                     'n_pca_components': n_comp}, f)
    print(f"\n  Saved pipeline to {OUT}/ridge_pipeline_{mkt}.pkl")

print(f"\n{'='*60}")
print("Done! Check outputs_ridge_pca/")


MODEL: Ridge Regression + PCA (Last Year Data)
Market: GDAM only
Period: 2025-03-01 onwards
Split:  Train≤2025-09-30  |  Val≤2025-11-30  |  Test>Val
PCA variance threshold: 0.995

──────────────────────────────────────────────────
Market: GDAM
──────────────────────────────────────────────────
  Rows: 35,040  (2025-03-01 → 2026-02-28)
  Train: 20,449 | Val: 5,856 | Test: 8,735
  Price mean — Train: 4657  Val: 3579  Test: 4256

  PCA: 32 features → 22 components (99.57% variance)
  Tuning alpha on PCA-transformed val set...
  Best alpha: 500.0  |  Val MAE: 567.9
  Results:
    Validation            MAE=  507.3  RMSE=  774.0  MAPE=18.06%
    Test                  MAE=  488.9  RMSE=  811.3  MAPE=10.84%

  Running residual analysis...
  Saved: outputs_ridge_pca/residual_analysis_GDAM.png

  Top-10 worst predicted days (by MAE):
    2026-01-12  MAE=957  avg_bias=-691 (UNDER)
    2026-01-05  MAE=793  avg_bias=-447 (UNDER)
    2025-12-22  MAE=762  avg_bias=-486 (UNDER)
    2026-01-03  MAE=759